# gorilla check on Colab (no Google Drive needed)

Runs the second-domain check (everyday numeric problems, generated by code) with the base model, with and without the gorilla pipeline. Nothing is read from or written to Drive, so there is no permission popup: results are printed here and stay in the session.

**Runtime → Change runtime type → T4 GPU** is enough for the 4B model (about 15 minutes in total).


In [ ]:
# Branch to test (an open PR's branch name, e.g. "colab-hf-sync"); keep "main" for normal use
BRANCH = "main"
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
!rm -rf /content/GENESIS-10T && git clone -q -b $BRANCH https://github.com/Joontark-kor/GENESIS-10T.git /content/GENESIS-10T
%cd /content/GENESIS-10T
!pip install -q -e ".[fin,qlora]" 2>&1 | tail -2
# Colab ships an old torchao that breaks peft; remove it
!pip uninstall -y -q torchao
!python -m genesis_calc.build_tasks --out data/applied/calc_v1 --per-kind 60 --seed 1 | head -3


## Baseline and gorilla on the same 420 questions

In [ ]:
!python -m genesis.harness.evaluate run --model Qwen/Qwen3-4B --template-kwargs '{"enable_thinking": false}' --data data/applied/calc_v1/eval_test.jsonl --name calc-base --batch-size 16 --max-new-tokens 128 --gpu-hour-price 1.0 --out reports/applied/calc-base.json 2>&1 | grep -v "Downloading\|Reconstruct\|Fetching\|Download complete"
!python -m genesis.harness.evaluate run --model Qwen/Qwen3-4B --template-kwargs '{"enable_thinking": false}' --data data/applied/calc_v1/eval_test.jsonl --name calc-gorilla --pipeline gorilla --batch-size 16 --max-new-tokens 128 --gpu-hour-price 1.0 --out reports/applied/calc-gorilla.json 2>&1 | grep -v "Downloading\|Reconstruct\|Fetching\|Download complete"
!python -m genesis.harness.evaluate compare reports/applied/calc-base.json reports/applied/calc-gorilla.json


## Wrong answers to read yourself

In [ ]:
import json
r = json.load(open("reports/applied/calc-gorilla.json"))
print("answered without a formula:", r["settings"].get("pipeline_fallbacks"), "of", len(r["rows"]))
for x in [x for x in r["rows"] if not x["correct"]][:5]:
    print(x["kind"], "gold:", x["gold"], "pred:", x["prediction"], "|", x["output"][-150:].replace("\n", " / "))


In [ ]:
# Keep the reports on the private Hugging Face repo so they survive a lost session (needs the HF_TOKEN secret).
# Failing here must not stop the notebook: the numbers are also printed above.
try:
    from genesis import hubsync
    TOKEN = hubsync.get_secret("HF_TOKEN")
    hubsync.sync_up(f"{hubsync.username(TOKEN)}/genesis-fin-v1-runs", "reports", TOKEN, path_in_repo="reports-gorilla")
    print("reports saved to the private Hugging Face repo")
except Exception as exc:
    print("NOT saved to Hugging Face:", type(exc).__name__)


## Optional: the finance set from the private Hugging Face dataset

Needs Kaggle/Colab Secret `HF_TOKEN` (notebook access on) and `<you>/genesis-fin-v1-data` holding `fin_v1.zip`. No upload dialog, no popup.


In [ ]:

from genesis import hubsync
TOKEN = hubsync.get_secret("HF_TOKEN")
hubsync.fetch_file(f"{hubsync.username(TOKEN)}/genesis-fin-v1-data", "fin_v1.zip", "/content", TOKEN)
!mkdir -p data/applied && unzip -o -q /content/fin_v1.zip -d data/applied
!python -m genesis.harness.evaluate run --model Qwen/Qwen3-4B --template-kwargs '{"enable_thinking": false}' --data data/applied/fin_v1/eval_test.jsonl --train-data data/applied/fin_v1/sft_train.jsonl --name fin-gorilla --pipeline gorilla --batch-size 16 --max-new-tokens 128 --gpu-hour-price 1.0 --out reports/applied/fin-gorilla.json 2>&1 | grep -v "Downloading\|Reconstruct\|Fetching\|Download complete"


In [ ]:
# Keep the reports on the private Hugging Face repo so they survive a lost session (needs the HF_TOKEN secret).
# Failing here must not stop the notebook: the numbers are also printed above.
try:
    from genesis import hubsync
    TOKEN = hubsync.get_secret("HF_TOKEN")
    hubsync.sync_up(f"{hubsync.username(TOKEN)}/genesis-fin-v1-runs", "reports", TOKEN, path_in_repo="reports-gorilla")
    print("reports saved to the private Hugging Face repo")
except Exception as exc:
    print("NOT saved to Hugging Face:", type(exc).__name__)
